# lora-appraisal-judge — E3 launcher (vLLM)

Scores the adapter trained in E2b, unchanged, on the test set rebuilt from the full
corpus (`data/e3/`: 12,691 examples, 319 tasks with both outcomes, 6,426 within-task
pairs, against 63 tasks and 106 pairs in E2b), with vLLM instead of one-by-one
Hugging Face generation. All the logic is in
[`scripts/evaluate_vllm.py`](https://github.com/carlosherreraUMA/lora-appraisal-judge/blob/master/scripts/evaluate_vllm.py).

**Settings (right-hand panel):** Accelerator → **GPU T4** · Internet → **On**.

**Input:** Add Input → Your Work → Notebooks → the **E2b training notebook**, the
version whose Output has `run/adapter/` and `run/test_scores.jsonl`. The script finds
both under `/kaggle/input` by itself.

**Order:** steps 0 and 1 interactively, read the smoke verdict, then step 2 (or
**Save & Run All** for an unattended run; `/kaggle/working` lands in the Output tab).

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/carlosherreraUMA/lora-appraisal-judge"
REPO_DIR = "/kaggle/working/repo"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "origin/master"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

# The commit being run: compare it with the latest one on GitHub if in doubt.
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %cd %s", "--date=short"], check=True)

In [ ]:
# Installs a vLLM whose torch pin matches the image's torch, then checks the stack
# imports cleanly and sees the GPU; it stops here, in seconds, if not.
# Re-running after a failed install: stop the session first (pip changes survive a
# kernel restart, not a new session).
!bash {REPO_DIR}/scripts/kaggle_setup_vllm.sh

## Step 1 — smoke test (about 10 minutes, mostly loading)

Checks that vLLM reproduces E2b's scores on 200 of E2b's own test examples (if it
does not, nothing else counts), times 500 E3 examples and projects the full run. It
tries vLLM's LoRA path first and, if that fails on the T4, the adapter merged into
the base weights, each in a fresh process. Ends with a **VERDICT**.

In [ ]:
!python -u {REPO_DIR}/scripts/evaluate_vllm.py --smoke --out /kaggle/working/e3-smoke

## Step 2 — full run

Refuses to start unless the smoke verdict was OK, and uses the mode that passed.
Agreement with E2b on all 1,717 examples first, then the E3 set; scores are appended
chunk by chunk, so re-running after an interruption resumes.

In [ ]:
!python -u {REPO_DIR}/scripts/evaluate_vllm.py --out /kaggle/working/e3 --after-smoke /kaggle/working/e3-smoke

## Results

`/kaggle/working/e3/results.json`: agreement with E2b, the E3 metrics for majority,
length and LoRA, the generated-label distribution and task-clustered bootstrap
intervals. Paste it back to Claude to fill in `EXPERIMENTS.md`.

In [ ]:
print(open("/kaggle/working/e3/results.json").read())